In [ ]:
%pip install pylatexenc
%pip install matplotlib
#%pip install git+https://github.com/It4innovations/quantum-as-a-service.git@main

# Connect to system

In [ ]:
# Setup Lexis session
from py4lexis.session import LexisSession

lexis_session = LexisSession()

In [ ]:
from qaas import QProvider, QBackend, QJob

token = lexis_session.get_access_token()
lexis_project = "vlq_demo_project"
resource_name = "qaas_user" 

provider = QProvider(token, lexis_project, resource_name)
# QBackend
backend:QBackend = provider.get_backend()

print(f'Qubit: {backend.architecture.qubits}')

print(f'Gates: {backend.architecture.gates.keys()}')

# Transpile Circuit

In [ ]:
from qiskit import QuantumCircuit
# Define quantum circuit
qc = QuantumCircuit.from_qasm_str("""
OPENQASM 2.0;
include "qelib1.inc";

qreg q[10];
qreg anc[7];
creg c[10];

// 1) Uniform superposition
h q[0]; h q[1]; h q[2]; h q[3]; h q[4];
h q[5]; h q[6]; h q[7]; h q[8]; h q[9];

// ===================== Grover iteration #1 =====================

// -------- ORACLE: mark |1011001101> (zeros at q[8], q[5], q[4], q[1]) --------
x q[8]; x q[5]; x q[4]; x q[1];

// Multi-controlled Z on all 10 qubits (via H–MCX–H on q[9])
h q[9];
ccx q[0], q[1], anc[0];
ccx anc[0], q[2], anc[1];
ccx anc[1], q[3], anc[2];
ccx anc[2], q[4], anc[3];
ccx anc[3], q[5], anc[4];
ccx anc[4], q[6], anc[5];
ccx anc[5], q[7], anc[6];
ccx anc[6], q[8], q[9];
ccx anc[5], q[7], anc[6];
ccx anc[4], q[6], anc[5];
ccx anc[3], q[5], anc[4];
ccx anc[2], q[4], anc[3];
ccx anc[1], q[3], anc[2];
ccx anc[0], q[2], anc[1];
ccx q[0], q[1], anc[0];
h q[9];

x q[8]; x q[5]; x q[4]; x q[1];

// -------- DIFFUSION (H^n X^n Z_111... X^n H^n) --------
h q[0]; h q[1]; h q[2]; h q[3]; h q[4];
h q[5]; h q[6]; h q[7]; h q[8]; h q[9];
x q[0]; x q[1]; x q[2]; x q[3]; x q[4];
x q[5]; x q[6]; x q[7]; x q[8]; x q[9];

h q[9];
ccx q[0], q[1], anc[0];
ccx anc[0], q[2], anc[1];
ccx anc[1], q[3], anc[2];
ccx anc[2], q[4], anc[3];
ccx anc[3], q[5], anc[4];
ccx anc[4], q[6], anc[5];
ccx anc[5], q[7], anc[6];
ccx anc[6], q[8], q[9];
ccx anc[5], q[7], anc[6];
ccx anc[4], q[6], anc[5];
ccx anc[3], q[5], anc[4];
ccx anc[2], q[4], anc[3];
ccx anc[1], q[3], anc[2];
ccx anc[0], q[2], anc[1];
ccx q[0], q[1], anc[0];
h q[9];

x q[0]; x q[1]; x q[2]; x q[3]; x q[4];
x q[5]; x q[6]; x q[7]; x q[8]; x q[9];
h q[0]; h q[1]; h q[2]; h q[3]; h q[4];
h q[5]; h q[6]; h q[7]; h q[8]; h q[9];

// ===================== Grover iteration #2 =====================

// ORACLE again (same marked state)
x q[8]; x q[5]; x q[4]; x q[1];

h q[9];
ccx q[0], q[1], anc[0];
ccx anc[0], q[2], anc[1];
ccx anc[1], q[3], anc[2];
ccx anc[2], q[4], anc[3];
ccx anc[3], q[5], anc[4];
ccx anc[4], q[6], anc[5];
ccx anc[5], q[7], anc[6];
ccx anc[6], q[8], q[9];
ccx anc[5], q[7], anc[6];
ccx anc[4], q[6], anc[5];
ccx anc[3], q[5], anc[4];
ccx anc[2], q[4], anc[3];
ccx anc[1], q[3], anc[2];
ccx anc[0], q[2], anc[1];
ccx q[0], q[1], anc[0];
h q[9];

x q[8]; x q[5]; x q[4]; x q[1];

// DIFFUSION again
h q[0]; h q[1]; h q[2]; h q[3]; h q[4];
h q[5]; h q[6]; h q[7]; h q[8]; h q[9];
x q[0]; x q[1]; x q[2]; x q[3]; x q[4];
x q[5]; x q[6]; x q[7]; x q[8]; x q[9];

h q[9];
ccx q[0], q[1], anc[0];
ccx anc[0], q[2], anc[1];
ccx anc[1], q[3], anc[2];
ccx anc[2], q[4], anc[3];
ccx anc[3], q[5], anc[4];
ccx anc[4], q[6], anc[5];
ccx anc[5], q[7], anc[6];
ccx anc[6], q[8], q[9];
ccx anc[5], q[7], anc[6];
ccx anc[4], q[6], anc[5];
ccx anc[3], q[5], anc[4];
ccx anc[2], q[4], anc[3];
ccx anc[1], q[3], anc[2];
ccx anc[0], q[2], anc[1];
ccx q[0], q[1], anc[0];
h q[9];

x q[0]; x q[1]; x q[2]; x q[3]; x q[4];
x q[5]; x q[6]; x q[7]; x q[8]; x q[9];
h q[0]; h q[1]; h q[2]; h q[3]; h q[4];
h q[5]; h q[6]; h q[7]; h q[8]; h q[9];

// ===================== End iterations =====================

// 4) Measure
measure q -> c;
"""
)

# simpler circuit
# qc = QuantumCircuit(2, 2)
# qc.id(0)  # Use identity gate instead
# qc.cz(0, 1)  # Use CZ instead of CNOT
# qc.measure([0, 1], [0, 1])

qc.draw(output='mpl')

In [ ]:
# Transpile circuit Locally
qc_transpiled = backend.transpile_to_IQM(qc, optimize_single_qubits=False, remote=False)

# Run Circuit via LEXIS Workflow

In [ ]:
# Run circuit with number of SHOTS
from py4lexis.core.typing.storage import Storage
from py4lexis.workflows.airflow import Airflow
from py4lexis.ddi.datasets import Datasets
from qiskit.qasm2 import dumps as dump_qasm2
import string
import random
import time
from datetime import datetime
from tempfile import NamedTemporaryFile, TemporaryDirectory
SHOTS = 1000


datasets = Datasets(session=lexis_session)

storages: list[Storage] = datasets.get_project_storages(project=lexis_project)
storage_name: str = storages[0]["storage_name"]
storage_resource: str = storages[0]["storage_resource"]


def generate_title(prefix="Transpiled_circuit_for_VLQ"):
    """Generates title to reference dataset with circuit

    :return: *{prefix}_{dt_str}_{rand_str}*
    """
    # Current datetime, e.g., "20250922_153045"
    dt_str = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Random 6-character string
    rand_str = ''.join(random.choices(
        string.ascii_letters + string.digits, k=6))
    return f"{prefix}_{dt_str}_{rand_str}"


qc_dataset_name = generate_title()
output_dataset_name = generate_title("Output_of_quantum_algorithm_from_VLQ")
workflow_run_id = generate_title("quantum_run_from_jupyter_for_VLQ")
file_name = ""


# Create temporary file and store transpiled circuit to be able to upload it to LEXIS dataset
with NamedTemporaryFile('w',suffix=".qasm2") as transpiled_qasm_circuit_file:
    transpiled_qasm_circuit_file.write(dump_qasm2(qc_transpiled))
    file_name = transpiled_qasm_circuit_file.name.split('/')[-1]
    datasets.tus_uploader_new(access="user",
                              project=lexis_project,
                              storage_name=storage_name,
                              storage_resource=storage_resource,
                              filename=file_name,
                              file_path="/".join(
                                  transpiled_qasm_circuit_file.name.split('/')[:-1]),
                              title=qc_dataset_name)
    dataset_with_transpiled_circuit = None
    while not dataset_with_transpiled_circuit: # Poll until dataset is ready
        datasets_by_title = datasets.get_all_datasets(title=qc_dataset_name)
        if len(datasets_by_title) > 0:
            dataset_with_transpiled_circuit = datasets_by_title[0]
        time.sleep(1.0)

qc_dataset_id = dataset_with_transpiled_circuit["dataset_id"]
print('Transpiled circuit dataset id: ', qc_dataset_id)

In [ ]:
# Define and run LEXIS workflow, which uploads results to dataset
airflow_mng = Airflow(session=lexis_session)
WORKFLOW_ID="Prototype_run_of_quantum_circuit"
dag_run = airflow_mng.execute_workflow(WORKFLOW_ID, {
    "job1": {
        # Input dataset with circuit
        "data_inputs": [
            {"source": {"info": {"path": f"/{file_name}", "dataset_id": qc_dataset_id}, "location": storage_name, "resource": storage_resource}, "target_path": "/"}],
        "data_outputs": [
            {"source_path": "/output", "target": {"info": {"access": "project", "additionalMetadata": {"Author": "QProvider user"}, "path": "/",
                                                           "title": output_dataset_name},
                                                  "location": storage_name, "resource": storage_resource}}],
        "requirements": {"environment_variables": {"QC_TRANSPILED":"TRUE"}, # When uploading not translated circuit, set FALSE or do not set
                         "job_name": "Run_of_quantum_circuit_from_jupyter",
                         "locations": [{"location_name": "VLQ", "location_resource": resource_name}],
                         "policy": "preferred", "project_shortname": lexis_project,
                         "subproject_identifier": lexis_project,
                         "tasks": [{"command_template_name": "run_quantum_circuit_simple",
                                    "max_cores": 4,
                                    "node_type_name": "",
                                    "template_parameter_values": {"RUN_SHOTS": SHOTS},
                                    "walltime_limit": 3600
                                    }]
                         }
    }
},
workflow_run_id=workflow_run_id
)
if dag_run:
    print("DAG run status: ", str(dag_run))
    print("DAG run id: ",workflow_run_id)
    print("QC dataset name: ", qc_dataset_name)
    print("Result output names: ", output_dataset_name)
else:
    print("Error occurred while executing airflow dag.")
    # Optionally a deletion of dataset with transpiled circuit is possible
    # datasets.delete_dataset_by_id(dataset_id=qc_dataset_id)

In [ ]:
# Wait for workflow and download results

workflow_status = dag_run['status']
while workflow_status in ['queued', 'running']: # Poll until workflow finish or fail
    states = airflow_mng.get_workflow_states(workflow_id=WORKFLOW_ID)
    state_found = False
    for state in states:
        if state['workflow_run_id'] == workflow_run_id:
            workflow_status=state['state']
            state_found = True
            break
    if not state_found:
        print("ERROR: Unable to find state of workflow")
    state_found = False
    time.sleep(1.0)


if workflow_status == 'success':
    with TemporaryDirectory("_vlq_jupyter") as out_directory:

        dataset_with_results = None
        datasets_by_title = datasets.get_all_datasets(title=output_dataset_name)
        if len(datasets_by_title) > 0:
            dataset_with_results = datasets_by_title[0]
        
        if dataset_with_results:
            # download dataset to temporary directory
            datasets.download_dataset(
                dataset_id=dataset_with_results['dataset_id'],
                result_name=output_dataset_name+".zip",
                destination_filepath=out_directory
                )
else:
    print("Workflow did not finished successfully!!!")

In [ ]:
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# Plot histogram
plot_histogram(results_dict, figsize=(12, 6), bar_labels=False)  # hide text above bars